## Finding example phrases and forms for annotation

Task: Annotate 1000 random obliques that appear in a locative case in the corpus with their semantic type. For proper annotation you need to see how the word is used in text. This notebook is for searching for usage examples for words from [a database file created on the basis of the Estonian Reference corpus](https://github.com/estnltk/syntax_experiments/tree/verb_templates/verb_transactions/v33). The examples aren't full sentences, but rather just a string containing a verb with all of its direct dependents.

The possible location words are from *kohasonad_test.csv*, which was made with *v01_locations_by_verb.ipynb*. Here we take 1000 random lemmas from that file to annotate. For ease of annotation we add the form and verbphrase the word appeared in.

In [1]:
import pandas as pd
import random
import sqlite3

In [2]:
# andmebaasi failinimi, kust andmeid loetake
filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

# andmebaasiga ühenduse loomine
conn = sqlite3.connect(filename)
cursor = conn.cursor()

In [3]:
#tagastab andmebaasist lemma, sõnavormi ja näitefraasi iga kasutaja määratud sõna kohta, mis kasutaja määratud verbiga esineb 
def kohad_naidetega(verbid):
    query = (f"SELECT lemma, transaction_row.form, phrase, sentence_id FROM `transaction_row` JOIN `transaction_head` ON transaction_head.id = transaction_row.head_id "
             f"WHERE verb IN ({','.join('?' for _ in verbid)}) AND lemma IN (SELECT lemma FROM temp_lemmad) AND transaction_row.deprel = 'obl' "
             f"AND (transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ?) ")
    cursor.execute(query, verbid + ('%adit%', '%ill%', '%in%', '%el%', '%all%', '%ad%', '%abl%'))
    kohad_naidetega = list(cursor.fetchall())
    return kohad_naidetega

In [5]:
#võtame umbes 20000 kohakäändes sõnast märgendamiseks 1000 random sõna
kohakaandes = pd.read_csv('C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\results\\kohasonad_test.csv')
sonad = kohakaandes['Unnamed: 0'].tolist() #words out
random.seed(10) #muuda numbrit kui tahad minust erinevat sampleit, default 10
sonad_1000 = random.sample(sonad, 1000) #1000 random words
sonad_1000 = tuple(sonad_1000)

In [6]:
#määrame verbid, mille alluvate seast sõnu ja näiteid otsitakse
verbid = ('jooksma', 'kõndima', 'tõttama', 'istuma', 'käima', 'astuma', 'liikuma', 'lahkuma')
#loome 1000 sõnaga ajutise tabeli, et sealt pärast päritavaid sõnu filtreerida
cursor.execute("CREATE TEMPORARY TABLE temp_lemmad (lemma TEXT)")
cursor.executemany("INSERT INTO temp_lemmad (lemma) VALUES (?)", [(lemma,) for lemma in sonad_1000])
#teostame andmebaasipäringu, tulemuseks list tupleitest kus on lemma, vorm ja näitefraas
naited = kohad_naidetega(verbid)
#kustutame ajutise lemmadega tabeli
cursor.execute("DROP TABLE temp_lemmad")

In [7]:
#paneme tulemused dataframei
naite_df = pd.DataFrame(naited, columns =['lemma', 'form', 'verbifraas', 'lause_id'])
#võtame iga sõna jaoks ainult esimese näite
naited_unique = naite_df.drop_duplicates(subset='lemma', keep='first')
naited_unique

,lemma,form,verbifraas,lause_id
0,Marko,Markole,tüdrukud on isa sõnul Markole hoogsalt amokki ...,775
1,poliitika,poliitikast,Lahkusin poliitikast aastal,880
2,ilme,ilmel,Nüüd käib kass ilmel aias ringi,2886
3,Kelam,Kelamitel,raske käib Kelamitel abiks vanadaam,2898
4,aastavahetus,aastavahetusel,Olen aastavahetusel koolides mängimas käinud,3305
...,...,...,...,...
8060,kalevispordihall,kalevispordihallis,raffast on kes käisid see kalevispordihallis,21138145
8064,päinaka,päinakas,slow sa käid päinakas,21178354
8071,aurama,auras,ma käis täna auras,21253362
8080,il,il,keegi RAHZEL il ka käis,21321966


In [8]:
#salvestame csv faili 
teekond = 'C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\results\\kohasonad_naited_lause_idga.csv'
naited_unique.to_csv(teekond)